In [46]:
# 📁 Standard Library Imports
import os      # Interact with the operating system (e.g., file paths, environment variables)
import json    # Parse and write JSON data (e.g., config files, chatbot intents)
import random  # Generate random numbers or select random elements (e.g., random responses)

# 🧠 NLP Toolkit
import nltk    # Natural Language Toolkit for text processing (tokenization, stemming, etc.)
nltk.download('wordnet')   # Download WordNet corpus for lemmatization
nltk.download('punkt')     # Download Punkt tokenizer for word tokenization

# ⚙️ PyTorch Core
import torch                # Core PyTorch library for tensor operations and GPU acceleration
import torch.nn as nn       # Neural network components (layers, loss functions, etc.)
import torch.nn.functional as F  # Functional API for activations, loss calculations, etc.
import torch.optim as optim       # Optimization algorithms (e.g., SGD, Adam)

# 📦 PyTorch Utilities
from torch.utils.data import DataLoader, TensorDataset  # Efficient data loading and batching


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\men_l\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\men_l\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [47]:
# Defines a neural network model for intent classification using PyTorch
class ChatbotModel(nn.Module):
    def __init__(self, input_size, output_size):
        # Calls the constructor of the parent nn.Module class to properly initialize the model
        super(ChatbotModel, self).__init__()

        # First fully connected layer: transforms input vector of size input_size to 128 hidden units
        self.fc1 = nn.Linear(input_size, 128)

        # Second fully connected layer: reduces 128 hidden units to 64
        self.fc2 = nn.Linear(128, 64)

        # Output layer: maps 64 hidden units to output_size (number of intent classes)
        self.fc3 = nn.Linear(64, output_size)

        # ReLU activation function: introduces non-linearity after each hidden layer
        self.relu = nn.ReLU()

        # Dropout layer: randomly zeroes out 50% of the neurons during training to prevent overfitting
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        # Applies first linear transformation followed by ReLU activation
        x = self.relu(self.fc1(x))

        # Applies dropout to reduce overfitting and improve generalization
        x = self.dropout(x)

        # Applies second linear transformation followed by ReLU activation
        x = self.relu(self.fc2(x))

        # Applies dropout again before final output layer
        x = self.dropout(x)

        # Applies final linear transformation to produce raw class scores (logits)
        x = self.fc3(x)

        # Returns the output logits for each intent class (to be passed to CrossEntropyLoss)
        return x


In [48]:
# Defines a class to manage chatbot logic, including intent parsing, training, inference, and response handling
class ChatbotAssistant:

    def __init__(self, intents_path, function_mappings=None):
        # Initializes the model placeholder; will be assigned after training or loading
        self.model = None

        # Stores the path to the JSON file containing chatbot intents and responses
        self.intents_path = intents_path

        # List to store tokenized training data as (pattern_words, intent_tag) pairs
        self.documents = []

        # List to store unique vocabulary words extracted from training patterns
        self.vocabulary = []

        # List to store unique intent tags
        self.intents = []

        # Dictionary mapping intent tags to their list of possible responses
        self.intents_responses = {}

        # Optional dictionary mapping intent tags to custom handler functions
        self.function_mappings = function_mappings

        # Placeholder for input feature matrix (bag-of-words vectors)
        self.X = None

        # Placeholder for output labels (intent indices)
        self.y = None

    @staticmethod
    def tokenize_and_lemmatize(text):
        # Initializes the WordNet lemmatizer from NLTK
        lemmatizer = nltk.WordNetLemmatizer()

        # Tokenizes the input text into individual words
        words = nltk.word_tokenize(text)

        # Converts each word to lowercase and lemmatizes it to its base form
        words = [lemmatizer.lemmatize(word.lower()) for word in words]

        # Returns the normalized list of words
        return words

    def bag_of_words(self, words):
        # Converts a list of words into a binary vector based on presence in the vocabulary
        return [1 if word in words else 0 for word in self.vocabulary]

    def parse_intents(self):
        # Initializes a lemmatizer (not used here directly, but could be reused)
        lemmatizer = nltk.WordNetLemmatizer()

        # Checks if the intents file exists
        if os.path.exists(self.intents_path):
            # Opens and loads the JSON file containing intents
            with open(self.intents_path, 'r') as f:
                intents_data = json.load(f)

            # Iterates through each intent object in the JSON
            for intent in intents_data['intents']:
                # Adds the intent tag if it's not already in the list
                if intent['tag'] not in self.intents:
                    self.intents.append(intent['tag'])

                    # Maps the tag to its list of response strings
                    self.intents_responses[intent['tag']] = intent['responses']

                # Processes each pattern associated with the intent
                for pattern in intent['patterns']:
                    # Tokenizes and lemmatizes the pattern text
                    pattern_words = self.tokenize_and_lemmatize(pattern)

                    # Adds the words to the vocabulary list
                    self.vocabulary.extend(pattern_words)

                    # Stores the (pattern_words, intent_tag) pair for training
                    self.documents.append((pattern_words, intent['tag']))

            # Deduplicates and sorts the vocabulary for consistent vectorization
            self.vocabulary = sorted(set(self.vocabulary))

    def prepare_data(self):
        # Initializes lists to hold input vectors and corresponding intent indices
        bags = []
        indices = []

        # Iterates through each training document
        for document in self.documents:
            words = document[0]  # Extracts tokenized words
            bag = self.bag_of_words(words)  # Converts to bag-of-words vector

            intent_index = self.intents.index(document[1])  # Gets index of intent tag

            bags.append(bag)  # Adds input vector
            indices.append(intent_index)  # Adds label index

        # Converts lists to NumPy arrays for model training
        self.X = np.array(bags)
        self.y = np.array(indices)

    def train_model(self, batch_size, lr, epochs):
        # Converts input and output data to PyTorch tensors
        X_tensor = torch.tensor(self.X, dtype=torch.float32)
        y_tensor = torch.tensor(self.y, dtype=torch.long)

        # Wraps tensors into a dataset object
        dataset = TensorDataset(X_tensor, y_tensor)

        # Creates a data loader to iterate over batches during training
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

        # Initializes the model with input size and number of intent classes
        self.model = ChatbotModel(self.X.shape[1], len(self.intents))

        # Defines the loss function for multi-class classification
        criterion = nn.CrossEntropyLoss()

        # Initializes the Adam optimizer with model parameters and learning rate
        optimizer = optim.Adam(self.model.parameters(), lr=lr)

        # Training loop over specified number of epochs
        for epoch in range(epochs):
            running_loss = 0.0  # Tracks cumulative loss

            # Iterates over each batch of training data
            for batch_X, batch_y in loader:
                optimizer.zero_grad()  # Clears previous gradients
                outputs = self.model(batch_X)  # Forward pass
                loss = criterion(outputs, batch_y)  # Computes loss
                loss.backward()  # Backpropagation
                optimizer.step()  # Updates weights
                running_loss += loss.item()  # Accumulates loss

            # Prints average loss for the epoch
            print(f"Epoch {epoch+1}: Loss: {running_loss / len(loader):.4f}")

    def save_model(self, model_path, dimensions_path):
        # Saves the model's trained weights to disk
        torch.save(self.model.state_dict(), model_path)

        # Saves input/output dimensions to a JSON file for future loading
        with open(dimensions_path, 'w') as f:
            json.dump({'input_size': self.X.shape[1], 'output_size': len(self.intents)}, f)

    def load_model(self, model_path, dimensions_path):
        # Loads input/output dimensions from the saved JSON file
        with open(dimensions_path, 'r') as f:
            dimensions = json.load(f)

        # Initializes the model with the saved dimensions
        self.model = ChatbotModel(dimensions['input_size'], dimensions['output_size'])

        # Loads the model weights from disk
        # ❌ FIX: 'weights_only=True' is invalid in PyTorch; remove it
        self.model.load_state_dict(torch.load(model_path))

    def process_message(self, input_message):
        # Tokenizes and lemmatizes the input message
        words = self.tokenize_and_lemmatize(input_message)

        # Converts the normalized words into a bag-of-words vector
        bag = self.bag_of_words(words)

        # Wraps the vector in a batch and converts to tensor
        bag_tensor = torch.tensor([bag], dtype=torch.float32)

        # Sets the model to evaluation mode
        self.model.eval()

        # Disables gradient tracking for inference
        with torch.no_grad():
            predictions = self.model(bag_tensor)  # Gets raw intent scores

        # Finds the index of the highest scoring intent
        predicted_class_index = torch.argmax(predictions, dim=1).item()

        # Maps the index to the corresponding intent tag
        predicted_intent = self.intents[predicted_class_index]

        # If a custom function is mapped to the predicted intent, execute it
        if self.function_mappings:
            if predicted_intent in self.function_mappings:
                self.function_mappings[predicted_intent]()  # Executes the function (return value ignored)

        # If responses are defined for the intent, return one randomly
        if self.intents_responses[predicted_intent]:
            return random.choice(self.intents_responses[predicted_intent])
        else:
            return None  # Returns None if no response is available


In [49]:
chatbot = ChatbotAssistant('intents.json')  
# Creates an instance of the ChatbotAssistant class, loading intents from the 'intents.json' file

chatbot.tokenize_and_lemmatize('Hello world how are you, I am programming in Python today')  
# Calls the static method to tokenize and lemmatize the input sentence for preprocessing


['hello',
 'world',
 'how',
 'are',
 'you',
 ',',
 'i',
 'am',
 'programming',
 'in',
 'python',
 'today']

In [50]:
chatbot.tokenize_and_lemmatize('run running runs ran')  
# Calls the static method to tokenize and lemmatize the input sentence for preprocessing

['run', 'running', 'run', 'ran']

In [ ]:
# Defines a function to randomly select and print 3 stock tickers from a predefined list
def get_stocks():
    stocks = ['APPL', 'META', 'NVDA', 'GS', 'MSFT']  # List of example stock symbols (note: 'APPL' should be 'AAPL' for Apple)

    print(random.sample(stocks, 3))  # Randomly selects and prints 3 unique stocks from the list

# Ensures the following block runs only when the script is executed directly (not imported as a module)
if __name__ == '__main__':
    # --- TRAINING BLOCK (commented out) ---
    # Initializes the chatbot assistant with the path to the intents file
    # Maps the 'stocks' intent to the get_stocks() function for dynamic response generation
    # assistant = ChatbotAssistant('intents.json', function_mappings={'stocks': get_stocks})

    # Parses the intents from the JSON file and prepares internal structures like vocabulary and documents
    # assistant.parse_intents()

    # Converts parsed documents into numerical training data (bag-of-words vectors and intent labels)
    # assistant.prepare_data()

    # Trains the chatbot model using the prepared data with specified batch size, learning rate, and number of epochs
    # assistant.train_model(batch_size=8, lr=0.001, epochs=100)

    # Saves the trained model weights and metadata (input/output dimensions) to disk for later use
    # assistant.save_model('chatbot_model.pth', 'dimensions.json')

    # --- INFERENCE BLOCK ---
    # Re-initializes the assistant for inference, mapping the 'stocks' intent to the get_stocks() function
    assistant = ChatbotAssistant('intents.json', function_mappings={'stocks': get_stocks})

    # Parses the intents and prepares internal mappings and vocabulary
    assistant.parse_intents()

    # Loads the trained model weights and input/output dimensions from disk
    assistant.load_model('chatbot_model.pth', 'dimensions.json')

    # Starts an infinite loop to interact with the user via command-line input
    while True:
        message = input('Enter your message:')  # Prompts the user to enter a message

        if message == '/quit':  # Checks if the user wants to exit the loop
            break  # Terminates the loop and ends the program

        # Processes the input message through the trained model and prints the chatbot's response
        print(assistant.process_message(message))
       

Enter your message: Hello


Good to see you again!


Enter your message: How are you?


Good to see you again!


Enter your message: What is MLOps?


Programming, coding or software development, means writing computer code to automate tasks.


Enter your message: What are my stocks?


['APPL', 'META', 'GS']
Here are your stocks!
